# 🌱 Sustainable Energy Consumption and Carbon Emission Analysis
### Machine Learning Prediction using Linear Regression
**Project by:** GTU Student | Semester 8
**Organization:** FICE Education Pvt. Ltd.

## Step 1 — Install & Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder

print('✅ All libraries loaded successfully!')

## Step 2 — Load Dataset

In [ ]:
# Upload energy_data_cleaned.csv to Colab first
df = pd.read_csv('energy_data_cleaned.csv')
print('Dataset Shape:', df.shape)
print('\nFirst 5 rows:')
df.head()

## Step 3 — Exploratory Data Analysis (EDA)

In [ ]:
print('Dataset Info:')
print(df.info())
print('\nBasic Statistics:')
df.describe()

In [ ]:
# CO2 Emission trend per country
plt.figure(figsize=(12,5))
for country in df['country'].unique():
    subset = df[df['country'] == country]
    plt.plot(subset['year'], subset['co2'], marker='o', label=country)

plt.title('CO₂ Emission Trend by Country (2010-2023)')
plt.xlabel('Year')
plt.ylabel('CO₂ Emission (Mt)')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('co2_trend.png', dpi=150)
plt.show()
print('✅ Chart saved as co2_trend.png')

In [ ]:
# Renewable vs Fossil Energy Share
latest = df[df['year'] == 2023]
x = np.arange(len(latest['country']))
width = 0.35

fig, ax = plt.subplots(figsize=(10,5))
ax.bar(x - width/2, latest['renewables_share_energy'], width, label='Renewables %', color='green')
ax.bar(x + width/2, latest['fossil_share_energy'], width, label='Fossil %', color='gray')
ax.set_xticks(x)
ax.set_xticklabels(latest['country'])
ax.set_title('Renewable vs Fossil Energy Share in 2023')
ax.set_ylabel('Share (%)')
ax.legend()
plt.tight_layout()
plt.savefig('energy_share.png', dpi=150)
plt.show()
print('✅ Chart saved as energy_share.png')

## Step 4 — Prepare Features for ML Model

In [ ]:
# Encode country as number
le = LabelEncoder()
df['country_encoded'] = le.fit_transform(df['country'])

# Features and Target
X = df[['year', 'energy_per_capita', 'renewables_share_energy', 'country_encoded']]
y = df['co2']

print('Features (X):')
print(X.head())
print('\nTarget (y - CO₂):')
print(y.head())

## Step 5 — Train Linear Regression Model

In [ ]:
# Split into train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train model
model = LinearRegression()
model.fit(X_train, y_train)

# Evaluate model
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print('✅ Model Trained Successfully!')
print(f'Mean Squared Error : {mse:.2f}')
print(f'R² Score           : {r2:.4f}')
print(f'Model Accuracy     : {r2*100:.2f}%')

## Step 6 — Predict Future CO₂ Emissions (2024–2035)

In [ ]:
countries = df['country'].unique()
predictions = []

for country in countries:
    country_data = df[df['country'] == country].iloc[-1]
    country_enc = le.transform([country])[0]
    for year in range(2024, 2036):
        years_ahead = year - 2023
        energy = country_data['energy_per_capita'] * (1 + 0.01 * years_ahead)
        renewables = min(country_data['renewables_share_energy'] + 0.5 * years_ahead, 80)
        pred = model.predict([[year, energy, renewables, country_enc]])[0]
        predictions.append({
            'country': country,
            'year': year,
            'predicted_co2': round(pred, 2),
            'energy_per_capita': round(energy, 2),
            'renewables_share_energy': round(renewables, 2)
        })

pred_df = pd.DataFrame(predictions)
pred_df.to_csv('prediction.csv', index=False)
print('✅ Predictions saved to prediction.csv')
pred_df.head(10)

## Step 7 — Actual vs Predicted CO₂ Chart

In [ ]:
plt.figure(figsize=(14,6))

colors = ['blue','red','green','orange','purple']
for i, country in enumerate(df['country'].unique()):
    actual = df[df['country'] == country]
    predicted = pred_df[pred_df['country'] == country]
    
    plt.plot(actual['year'], actual['co2'], 
             color=colors[i], linewidth=2, label=f'{country} (Actual)')
    plt.plot(predicted['year'], predicted['predicted_co2'], 
             color=colors[i], linewidth=2, linestyle='--', label=f'{country} (Predicted)')

plt.axvline(x=2023, color='black', linestyle=':', linewidth=1.5, label='Prediction Start')
plt.title('Actual vs Predicted CO₂ Emissions (2010–2035)')
plt.xlabel('Year')
plt.ylabel('CO₂ Emission (Mt)')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
plt.grid(True)
plt.tight_layout()
plt.savefig('actual_vs_predicted.png', dpi=150)
plt.show()
print('✅ Chart saved as actual_vs_predicted.png')

## ✅ Summary
- Dataset: 5 countries, 2010–2023
- Model: Linear Regression
- Features: Year, Energy per Capita, Renewables Share
- Target: CO₂ Emissions
- Output: Predictions from 2024–2035

**Files generated:**
- `prediction.csv` → import into Power BI
- `co2_trend.png` → use in project report
- `energy_share.png` → use in project report
- `actual_vs_predicted.png` → use in project report & PPT